~~~
Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
~~~

# Evaluate for medical question answering (MedQA)

<table><tbody><tr>
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/google-health/medgemma/blob/main/notebooks/evaluate_on_medqa.ipynb">
      <img alt="Google Colab logo" src="https://www.tensorflow.org/images/colab_logo_32px.png" width="32px"><br> Run in Google Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogle-Health%2Fmedgemma%2Fmain%2Fnotebooks%2Fevaluate_on_medqa.ipynb">
      <img alt="Google Cloud Colab Enterprise logo" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" width="32px"><br> Run in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/google-health/medgemma/blob/main/notebooks/evaluate_on_medqa.ipynb">
      <img alt="GitHub logo" src="https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png" width="32px"><br> View on GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://huggingface.co/collections/google/medgemma-release-680aade845f90bec6a3f60c4">
      <img alt="Hugging Face logo" src="https://huggingface.co/front/assets/huggingface_logo-noborder.svg" width="32px"><br> View on Hugging Face
    </a>
  </td>
</tr></tbody></table>

This notebook provides a basic demo of using MedGemma for MedQA evaluation using [vLLM](https://docs.vllm.ai/en/latest/). It loads the model locally, processes the MedQA dataset, and evaluates model accuracy.

This notebook demonstrates how to evaluate MedGemma on the MedQA dataset using vLLM for high-performance inference. It covers the end-to-end workflow of loading the model, processing the dataset, and measuring accuracy. It uses a robust output parser and sets max_tokens to a high value (e.g., 10,000) to accommodate the model's detailed reasoning traces.

## Setup

To complete this tutorial, you'll need to have a runtime with [sufficient resources](https://ai.google.dev/gemma/docs/core#sizes) to run the MedGemma 1.5 4B model.

You can try out this evaluation in Google Colab using an L4 GPU:

1. In the upper-right of the Colab window, select **▾ (Additional connection options)**.
2. Select **Change runtime type**.
3. Under **Hardware accelerator**, select **L4 GPU**.

### Get access to MedGemma

Before you get started, make sure that you have access to MedGemma models on Hugging Face:

1. If you don't already have a Hugging Face account, you can create one for free by clicking [here](https://huggingface.co/join).
2. Head over to the [MedGemma model page](https://huggingface.co/google/medgemma-1.5-4b-it) and accept the usage conditions.

### Authenticate with Hugging Face

Generate a Hugging Face `read` access token by going to [settings](https://huggingface.co/settings/tokens).

If you are using Google Colab, add your access token to the Colab Secrets manager to securely store it. If not, proceed to run the cell below to authenticate with Hugging Face.

1. Open your Google Colab notebook and click on the 🔑 Secrets tab in the left panel.
2. Create a new secret with the name `HF_TOKEN`.
3. Copy/paste your token key into the Value input box of `HF_TOKEN`.
4. Toggle the button on the left to allow notebook access to the secret.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT"):
    # Use secret if running in Google Colab
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Store Hugging Face data under `/content` if running in Colab Enterprise
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    # Authenticate with Hugging Face
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

In [ ]:
# Install dependencies
!pip install -q transformers==4.57.3 vllm==0.17.1 datasets

## Load model with vLLM

This section demonstrates how to load the MedGemma model using vLLM for high-performance inference.

In [ ]:
# Load model with vLLM
from vllm import LLM, SamplingParams

model_id = "google/medgemma-1.5-4b-it"

# Set max_model_len to a smaller value (e.g., 10000) to prevent Out of Memory errors
llm = LLM(model=model_id, max_model_len=10000)
sampling_params = SamplingParams(max_tokens=10000, temperature=0.0) # temperature=0.0 for deterministic generation

## Run inference and score results on MedQA

This section demonstrates how to load the MedQA dataset, format prompts, run inference in batches using vLLM, and evaluate the model's accuracy.

In [ ]:
# Run inference and score results

import datasets
import re

# Load dataset
print("Loading MedQA dataset from openlifescienceai/medqa...")
medqa_dataset = datasets.load_dataset("openlifescienceai/medqa")
test_dataset = medqa_dataset["test"]
# # subset
# test_dataset = test_dataset.select(range(100))
print(f"Loaded {len(test_dataset)} test examples.")

def format_prompt(x):
    prompt_template = """Answer the given question. Think step by step.
You can directly provide the answer (A single letter), without further additions. E.g. "Final Answer: (A)".
Question: {question}
{options}
"""
    options = x['data']['Options']
    options_str = f"(A) {options['A']} (B) {options['B']} (C) {options['C']} (D) {options['D']}"
    return {
        'prompt': prompt_template.format(question=x['data']['Question'], options=options_str),
        'answer': x['data']['Correct Option']
    }

# Process dataset
processed_test = test_dataset.map(format_prompt)

# Parsing logic
ANSWER_PATTERNS = [
    r'The final answer is\s\(([A-J])\)', r'The final answer is\s\**\(([A-J])\)\**',
    r'The final answer is\s\$\\boxed{([A-J])}\$', r'Final Answer:\(([A-J])\)',
    r'Final Answer:\s\(([A-J])\)', r'Final Answer:\s\(?([A-J])',
    r'Final Answer:\s*\**\(([A-J])\)\**', r'\**Final Answer:\**\s\(([A-J])\)',
]

def extract_answer(text):
    if not isinstance(text, str) or not text:
        return None
    for pat in ANSWER_PATTERNS:
        m = re.search(pat, text)
        if m:
            return m.group(1)
    return None

# Run inference with vLLM in batches and evaluate
tokenizer = llm.get_tokenizer()
all_prompts = []

for item in processed_test:
    messages = [
        {'role': 'system', 'content': 'SYSTEM INSTRUCTION: think silently if needed.'},
        {'role': 'user', 'content': item['prompt']}
    ]
    # Convert chat messages to a single string prompt using the model's chat template
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    all_prompts.append(prompt)

all_outputs = []
print(f"Running vLLM inference on {len(all_prompts)} examples...")

all_outputs = llm.generate(all_prompts, sampling_params)

correct = 0
total = len(processed_test)

print("Evaluating results...")

for i, (item, output) in enumerate(zip(processed_test, all_outputs)):
    response = output.outputs[0].text

    extracted = extract_answer(response)
    is_correct = extracted == item['answer']

    if is_correct:
        correct += 1

    if i % 10 == 0: # Print progress every 10 examples
        print(f"Processed {i}/{total} examples...")
        print(f"Expected: {item['answer']}, Extracted: {extracted}")
        print(f"Correct: {is_correct}")
        print("-" * 20)

accuracy = correct / total if total > 0 else 0
print(f"Total Accuracy: {accuracy:.2%} ({correct}/{total})")


Loading MedQA dataset from openlifescienceai/medqa...
Loaded 1273 test examples.


Map:   0%|          | 0/1273 [00:00<?, ? examples/s]

Running vLLM inference on 1273 examples...


Rendering prompts:   0%|          | 0/1273 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1273 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Evaluating results...
Processed 0/1273 examples...
Expected: B, Extracted: A
Correct: False
--------------------
Processed 10/1273 examples...
Expected: D, Extracted: D
Correct: True
--------------------
Processed 20/1273 examples...
Expected: C, Extracted: C
Correct: True
--------------------
Processed 30/1273 examples...
Expected: D, Extracted: B
Correct: False
--------------------
Processed 40/1273 examples...
Expected: C, Extracted: C
Correct: True
--------------------
Processed 50/1273 examples...
Expected: D, Extracted: D
Correct: True
--------------------
Processed 60/1273 examples...
Expected: C, Extracted: C
Correct: True
--------------------
Processed 70/1273 examples...
Expected: A, Extracted: A
Correct: True
--------------------
Processed 80/1273 examples...
Expected: B, Extracted: B
Correct: True
--------------------
Processed 90/1273 examples...
Expected: D, Extracted: D
Correct: True
--------------------
Processed 100/1273 examples...
Expected: D, Extracted: B
Correct: F